---
# 03. FPI 산출

## 분석 개요

버거+샌드위치+패스트푸드 업종 독립 브랜드를 대상으로
주변 프랜차이즈의 경쟁압력을 수치화한 FPI를 산출한다.

restaurant과 동일한 방법론을 적용하되, 분석 대상을 버거+샌드+패스트푸드로 한정한다.

| 단계 | 내용 |
|---|---|
| STEP 3-0 | 감성점수 Shift 정규화 |
| STEP 3-1 | 거리 계산 (하버사인 공식) |
| STEP 3-2 | FPI 산출 (반경별) |
| STEP 3-3 | 민감도 분석 (임계거리 확정) |
| STEP 3-4 | FPI 구간 분류 (NP/LP/HP) |
| STEP 3-5 | 저장 |

**FPI 공식**

$$FPI = \sum \frac{(\text{별점} \times \ln(\text{리뷰수}+1) \times \text{감성점수}_{shifted})}{(\text{거리(km)}+k)^2}$$

**입력 데이터**
- `biz_sentiment_burger.csv`: 업체별 감성점수

**출력 데이터**
- `biz_indie_with_groups_burger.csv`: FPI 구간 분류 완료

## 공통 라이브러리 및 설정

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

PATH_to_data = "../results"
PATH_to_save = "../results"

# 데이터 로드
df = pd.read_csv(f"{PATH_to_data}/biz_sentiment_burger.csv")

indie_df = df[df['is_franchise'] == False].copy()
fran_df  = df[df['is_franchise'] == True].copy()

print(f"독립 브랜드: {len(indie_df):,}개")
print(f"프랜차이즈 : {len(fran_df):,}개")
print(f"\n프랜차이즈 감성점수 분포:")
print(fran_df['tfidf_sentiment'].describe().round(4))
print(f"\n음수 감성점수 프랜차이즈: {(fran_df['tfidf_sentiment'] < 0).sum()}개")

독립 브랜드: 775개
프랜차이즈 : 804개

프랜차이즈 감성점수 분포:
count    804.0000
mean      -0.0108
std        0.0163
min       -0.1109
25%       -0.0216
50%       -0.0111
75%        0.0005
max        0.0475
Name: tfidf_sentiment, dtype: float64

음수 감성점수 프랜차이즈: 594개


---
## STEP 3-0. 감성점수 Shift 정규화

프랜차이즈 감성점수의 음수값이 FPI를 음수로 만드는 문제를 방지하기 위해
최솟값을 빼서 전체를 양수 범위로 이동시킨다.
값들 간의 상대적 간격과 절대적 크기는 그대로 유지된다.

팀플과 동일한 방식 적용.

In [2]:
# Shift 정규화
sentiment_min = fran_df['tfidf_sentiment'].min()
fran_df['sentiment_shifted'] = fran_df['tfidf_sentiment'] - sentiment_min

print(f"감성점수 최솟값 (shift 기준): {sentiment_min:.4f}")
print(f"\nShift 정규화 전후 비교:")
print(f"{'':15} {'최솟값':>10} {'최댓값':>10} {'평균':>10}")
print(f"{'정규화 전':15} {fran_df['tfidf_sentiment'].min():>10.4f} "
      f"{fran_df['tfidf_sentiment'].max():>10.4f} "
      f"{fran_df['tfidf_sentiment'].mean():>10.4f}")
print(f"{'정규화 후':15} {fran_df['sentiment_shifted'].min():>10.4f} "
      f"{fran_df['sentiment_shifted'].max():>10.4f} "
      f"{fran_df['sentiment_shifted'].mean():>10.4f}")
print(f"\n음수값 잔존 여부: {(fran_df['sentiment_shifted'] < 0).sum()}개")

감성점수 최솟값 (shift 기준): -0.1109

Shift 정규화 전후 비교:
                       최솟값        최댓값         평균
정규화 전              -0.1109     0.0475    -0.0108
정규화 후               0.0000     0.1584     0.1001

음수값 잔존 여부: 0개


---
## STEP 3-1. 거리 계산

모든 독립 브랜드와 프랜차이즈 간의 물리적 거리를 하버사인 공식으로 계산한다.

In [3]:
print("독립 브랜드 - 프랜차이즈 간 거리 계산 중...")

indie_lat = np.radians(indie_df['latitude'].values)[:, np.newaxis]
indie_lon = np.radians(indie_df['longitude'].values)[:, np.newaxis]
fran_lat  = np.radians(fran_df['latitude'].values)[np.newaxis, :]
fran_lon  = np.radians(fran_df['longitude'].values)[np.newaxis, :]

dlat = indie_lat - fran_lat
dlon = indie_lon - fran_lon
a = np.sin(dlat/2)**2 + np.cos(indie_lat) * np.cos(fran_lat) * np.sin(dlon/2)**2
distances_m = 2 * np.arcsin(np.sqrt(a)) * 6371000

print(f"거리 행렬 shape: {distances_m.shape}")
print(f"  ({len(indie_df):,}개 독립 브랜드 × {len(fran_df):,}개 프랜차이즈)")

독립 브랜드 - 프랜차이즈 간 거리 계산 중...
거리 행렬 shape: (775, 804)
  (775개 독립 브랜드 × 804개 프랜차이즈)


---
## STEP 3-2. FPI 산출

반경 100m / 200m / 300m / 500m 4개 기준으로 FPI를 산출한다.
보정 상수 k=1 적용 (거리 km 단위, 극단값 발산 방지).

In [4]:
# 프랜차이즈 분자 계산
fran_scores = (fran_df['stars'].values
               * np.log(fran_df['review_count'].values + 1)
               * fran_df['sentiment_shifted'].values)

print(f"프랜차이즈 분자 점수 확인:")
print(f"  최솟값: {fran_scores.min():.4f}")
print(f"  최댓값: {fran_scores.max():.4f}")
print(f"  음수 여부: {(fran_scores < 0).sum()}개")

radii = [100, 200, 300, 500]
k = 1
distances_km = distances_m / 1000.0

print(f"\n반경별 FPI 계산 중...")
for r in radii:
    mask = distances_m <= r
    fpi_terms = fran_scores[np.newaxis, :] / (distances_km + k)**2
    fpi_sums  = np.sum(np.where(mask, fpi_terms, 0.0), axis=1)
    indie_df[f'fpi_{r}m'] = fpi_sums
    print(f"  fpi_{r}m 완료 | 음수: {(fpi_sums < 0).sum()}개 | 0초과: {(fpi_sums > 0).sum()}개")

프랜차이즈 분자 점수 확인:
  최솟값: 0.0000
  최댓값: 4.1848
  음수 여부: 0개

반경별 FPI 계산 중...
  fpi_100m 완료 | 음수: 0개 | 0초과: 293개
  fpi_200m 완료 | 음수: 0개 | 0초과: 511개
  fpi_300m 완료 | 음수: 0개 | 0초과: 592개
  fpi_500m 완료 | 음수: 0개 | 0초과: 672개


---
## STEP 3-3. 민감도 분석

반경별 FPI가 독립 브랜드의 별점과 감성점수에 미치는 영향을
단순 회귀분석으로 비교하여 최적 임계거리를 도출한다.

**종속변수 구성**
- 별점 (stars)
- 감성점수 (tfidf_sentiment)
- 복합지수 (별점 × log 리뷰수) (추가)
- 폐업여부 (is_open) — 로지스틱 회귀 (추가)

> **restaurant과의 차이**: 버거+샌드+패스트푸드 업종에서는
> 별점만 FPI에 유의미하게 반응하였으며,
> 감성점수·복합지수·폐업여부는 단순 회귀에서 유의미하지 않았다.
> 이는 통제변수 없이 FPI 단독으로는 별점 외 종속변수를 설명하기 어려움을 시사한다.
> 폐업여부는 STEP 4에서 별점·리뷰수·상권을 통제한 후 재분석한다.

In [7]:
# 복합지수 생성
indie_df['weighted_score'] = (
    indie_df['stars'] * np.log(indie_df['review_count'] + 1)
)

print("종속변수 기술통계:")
print(f"\n별점:")
print(indie_df['stars'].describe().round(2))
print(f"\nlog(리뷰수+1):")
print(np.log(indie_df['review_count']+1).describe().round(2))
print(f"\n복합지수 (별점 × log 리뷰수):")
print(indie_df['weighted_score'].describe().round(2))
print(f"\n폐업 여부 분포:")
print(indie_df['is_open'].value_counts())

print("\n" + "=" * 60)
print("[민감도 분석] 별점(Stars)에 미치는 영향")
print("=" * 60)
rows = []
for r in radii:
    X = sm.add_constant(indie_df[f'fpi_{r}m'])
    model = sm.OLS(indie_df['stars'], X).fit()
    rows.append({'반경': f'{r}m',
                 '회귀계수': round(model.params.iloc[1], 4),
                 'P-value': round(model.pvalues.iloc[1], 4),
                 'R²': round(model.rsquared, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n" + "=" * 60)
print("[민감도 분석] 감성점수(Sentiment)에 미치는 영향")
print("=" * 60)
rows = []
for r in radii:
    X = sm.add_constant(indie_df[f'fpi_{r}m'])
    model = sm.OLS(indie_df['tfidf_sentiment'], X).fit()
    rows.append({'반경': f'{r}m',
                 '회귀계수': round(model.params.iloc[1], 4),
                 'P-value': round(model.pvalues.iloc[1], 4),
                 'R²': round(model.rsquared, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n" + "=" * 60)
print("[민감도 분석] 복합지수(별점 × log 리뷰수)에 미치는 영향")
print("=" * 60)
rows = []
for r in radii:
    X = sm.add_constant(indie_df[f'fpi_{r}m'])
    model = sm.OLS(indie_df['weighted_score'], X).fit()
    rows.append({'반경': f'{r}m',
                 '회귀계수': round(model.params.iloc[1], 4),
                 'P-value': round(model.pvalues.iloc[1], 4),
                 'R²': round(model.rsquared, 4)})
print(pd.DataFrame(rows).to_string(index=False))

print("\n" + "=" * 60)
print("[민감도 분석] 폐업여부(is_open)에 미치는 영향 (로지스틱)")
print("=" * 60)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

rows = []
for r in radii:
    X_lr = indie_df[[f'fpi_{r}m']].copy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_lr)
    y_lr = indie_df['is_open']
    
    lr = LogisticRegression()
    lr.fit(X_scaled, y_lr)
    
    # p-value는 statsmodels로
    X_sm = sm.add_constant(indie_df[f'fpi_{r}m'])
    logit = sm.Logit(y_lr, X_sm).fit(disp=0)
    rows.append({'반경': f'{r}m',
                 '회귀계수': round(logit.params.iloc[1], 4),
                 'P-value': round(logit.pvalues.iloc[1], 4),
                 'Odds Ratio': round(np.exp(logit.params.iloc[1]), 4)})
print(pd.DataFrame(rows).to_string(index=False))

종속변수 기술통계:

별점:
count    775.00
mean       3.55
std        0.77
min        1.00
25%        3.00
50%        3.50
75%        4.00
max        5.00
Name: stars, dtype: float64

log(리뷰수+1):
count    775.00
mean       4.10
std        1.47
min        1.39
25%        2.97
50%        4.20
75%        5.17
max        8.86
Name: review_count, dtype: float64

복합지수 (별점 × log 리뷰수):
count    775.00
mean      14.72
std        6.51
min        1.61
25%        9.52
50%       14.15
75%       19.26
max       38.21
Name: weighted_score, dtype: float64

폐업 여부 분포:
is_open
1    535
0    240
Name: count, dtype: int64

[민감도 분석] 별점(Stars)에 미치는 영향
  반경    회귀계수  P-value     R²
100m -0.0419   0.1226 0.0031
200m -0.0149   0.3781 0.0010
300m -0.0299   0.0147 0.0077
500m -0.0291   0.0014 0.0132

[민감도 분석] 감성점수(Sentiment)에 미치는 영향
  반경    회귀계수  P-value     R²
100m -0.0009   0.1265 0.0030
200m -0.0002   0.5718 0.0004
300m -0.0004   0.1395 0.0028
500m -0.0002   0.1914 0.0022

[민감도 분석] 복합지수(별점 × log 리뷰수)에 미치는 영향
  반경    회귀계수 

---
## STEP 3-4. FPI 구간 분류

민감도 분석 결과를 바탕으로 임계거리를 확정하고
FPI 구간(NP/LP/HP)을 분류한다.

- NP (무풍지대): fpi = 0
- LP (저압력): fpi > 0, 중앙값 이하
- HP (고압력): fpi > 0, 중앙값 초과

In [8]:
# 민감도 분석 결과 확인 후 임계거리 입력
RADIUS = 300  # 민감도 분석 결과에 따라 수정

indie_fpi = indie_df.copy()
fpi_col = f'fpi_{RADIUS}m'

fpi_nonzero = indie_fpi[indie_fpi[fpi_col] > 0][fpi_col]
median_fpi  = fpi_nonzero.median()

def assign_group(fpi):
    if fpi == 0:
        return 'NP'
    elif fpi <= median_fpi:
        return 'LP'
    else:
        return 'HP'

indie_fpi['fpi_group'] = indie_fpi[fpi_col].apply(assign_group)

print(f"임계거리: {RADIUS}m")
print(f"중앙값 기준 (fpi_{RADIUS}m > 0): {median_fpi:.4f}")
print(f"\nFPI 구간 분포:")
print(indie_fpi['fpi_group'].value_counts())
print(f"\nNaN 여부: {indie_fpi['fpi_group'].isna().sum()}개")

# 구간별 평균 별점
print(f"\n구간별 평균 별점:")
print(indie_fpi.groupby('fpi_group')['stars'].mean().round(2))

임계거리: 300m
중앙값 기준 (fpi_300m > 0): 1.9327

FPI 구간 분포:
fpi_group
HP    296
LP    296
NP    183
Name: count, dtype: int64

NaN 여부: 0개

구간별 평균 별점:
fpi_group
HP    3.46
LP    3.53
NP    3.70
Name: stars, dtype: float64


---
## STEP 3-5. 저장

In [9]:
# 원본 데이터에 FPI 결합
final_df = pd.merge(
    df,
    indie_fpi[['business_id'] + [f'fpi_{r}m' for r in radii] + ['fpi_group']],
    on='business_id', how='left'
)
for r in radii:
    final_df[f'fpi_{r}m'] = final_df[f'fpi_{r}m'].fillna(0)

# 독립 브랜드만 저장
indie_final = final_df[final_df['is_franchise'] == False].copy()
indie_final.to_csv(f"{PATH_to_save}/biz_indie_with_groups_burger.csv",
                   index=False, encoding='utf-8-sig')

print(f"저장 완료: biz_indie_with_groups_burger.csv")
print(f"shape: {indie_final.shape}")
print(f"\nFPI 음수 잔존 확인:")
for r in radii:
    neg = (indie_final[f'fpi_{r}m'] < 0).sum()
    print(f"  fpi_{r}m 음수: {neg}개")

저장 완료: biz_indie_with_groups_burger.csv
shape: (775, 21)

FPI 음수 잔존 확인:
  fpi_100m 음수: 0개
  fpi_200m 음수: 0개
  fpi_300m 음수: 0개
  fpi_500m 음수: 0개


---
## STEP 3 결과 정리

### 민감도 분석 결과

| 반경 | 별점 p-value | 감성점수 p-value | 복합지수 p-value | 폐업 p-value |
|---|---|---|---|---|
| 100m | 0.1226 ❌ | 0.1265 ❌ | 0.8741 ❌ | 0.7870 ❌ |
| 200m | 0.3781 ❌ | 0.5718 ❌ | 0.2465 ❌ | 0.9583 ❌ |
| **300m** | **0.0147 ✅** | 0.1395 ❌ | 0.7495 ❌ | 0.2272 ❌ |
| 500m | 0.0014 ✅ | 0.1914 ❌ | 0.6812 ❌ | 0.1842 ❌ |

**임계거리 확정: 300m**

restaurant과 동일한 임계거리. 별점 기준 p<0.05 최소 반경.

**restaurant과의 차이**

- restaurant: 300m에서 별점(p=0.000) + 감성점수(p=0.021) 모두 유의미
- 버거: 300m에서 별점(p=0.015)만 유의미, 감성점수 전 반경 유의미하지 않음
- 해석: 버거+샌드+패스트푸드 업종의 리뷰 언어는 경쟁 환경보다
  개별 매장 경험에 더 민감하게 반응하는 것으로 해석된다.

---

### FPI 구간 분포 (300m 기준)

| 구간 | 업체 수 | 비율 | 평균 별점 |
|---|---|---|---|
| NP (무풍지대) | 183개 | 23.6% | 3.70 |
| LP (저압력) | 296개 | 38.2% | 3.53 |
| HP (고압력) | 296개 | 38.2% | 3.46 |

NP > LP > HP 순으로 평균 별점이 하향하는 패턴이 확인된다.

restaurant(NP 3.83 > LP 3.69 > HP 3.61)과 동일한 방향의 결과다.

---

### 04에서 추가 분석할 것

단순 회귀에서 유의미하지 않았던 폐업여부를
별점·리뷰수·상권을 통제변수로 투입한 로지스틱 회귀로 재분석한다.

> "별점이 같은 조건에서도 FPI가 높으면 폐업 확률이 높아지는가?"